## This notebook documents my progress learning memo. 
- Currently it models a single timestep of an agent in a grid taking actions towards its goals.  
- I am not sure if memo is suitable for the sequential nature of gridworld. Might need to use snapshots_self_as? 
- Currently I am confusing forward planning and inverse planning into one agent model. It shouldn't be "choosing" a goal for forward planning, it should know the goal. 

In [235]:
# Import dependencies
import jax
import jax.numpy as np
from memo import memo
from enum import Enum
import matplotlib.pyplot as plt

In [236]:
from enum import IntEnum

class Actions(IntEnum):
    STAY = 0 
    UP = 1
    DOWN = 2
    LEFT = 3
    RIGHT = 4

actions_dict = {
    Actions.STAY: np.array([0, 0]), 
    Actions.UP: np.array([0, 1]), 
    Actions.DOWN: np.array([0, -1]), 
    Actions.LEFT: np.array([-1, 0]), 
    Actions.RIGHT: np.array([1, 0])
}

In [237]:
# Define a 5x5 grid where 1s are walls
grid = np.array(
    [[0,0,0,0,0],
    [0,1,1,0,1],
    [0,0,0,0,0],
    [0,1,0,1,0],
    [0,1,0,1,0]])

# Define goal locations
goal_coords = np.array([(2, 2), (2, 4), (4, 2)])  # 3 possible goal locations (all free cells)
Goals = range(len(goal_coords))  # Indices: 0, 1, 2

In [238]:
# Not using this right now as the steps are implicit in compute distance table
@jax.jit
def step(grid, state, action):
    next_state = state + action
    # Check boundaries
    rows, cols = grid.shape
    in_bounds = np.all(
        (next_state >= 0) & (next_state < np.array([rows, cols]))
    )
    return np.where(in_bounds, next_state, state)

In [239]:
from collections import deque

# Pre-compute distances (run once in Python, not JAX)
def compute_distance_table(grid):
    """Compute shortest path distances from every location to every other location."""
    h, w = grid.shape
    # distances[goal_y, goal_x, loc_y, loc_x] = distance from loc to goal
    distances = np.full((h, w, h, w), np.inf)
    
    # Run Dijkstra from each possible goal
    for goal_y in range(h):
        for goal_x in range(w):
            if grid[goal_y, goal_x]:  # Skip if wall
                continue
            
            dist = np.full((h, w), np.inf)
            dist = dist.at[goal_y, goal_x].set(0.0)
            queue = deque([(0.0, (goal_y, goal_x))])
            visited = set()
            
            while queue:
                d, (y, x) = queue.popleft()
                if (y, x) in visited:
                    continue
                visited.add((y, x))
                
                for dy, dx in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < h and 0 <= nx < w and not grid[ny, nx]:
                        new_dist = d + 1.0
                        if new_dist < dist[ny, nx]:
                            dist = dist.at[ny, nx].set(new_dist)
                            queue.append((new_dist, (ny, nx)))
            
            distances = distances.at[goal_y, goal_x].set(dist)
    
    return distances

# Compute once and store
distances = compute_distance_table(grid)

In [240]:
# Define q_value to capture distances from global scope (closure)
@jax.jit
def q_value(state_y, state_x, goal_idx, action):
    # Get goal coordinates from index
    goal_y = goal_coords[goal_idx][0]
    goal_x = goal_coords[goal_idx][1]
    
    # Actions: STAY=0, UP=1, DOWN=2, LEFT=3, RIGHT=4
    # Coordinates are (row, col) where row increases downward
    # UP decreases row, DOWN increases row
    action_deltas = np.array([
        [0, 0],   # STAY
        [-1, 0],  # UP: decrease row
        [1, 0],   # DOWN: increase row  
        [0, -1],  # LEFT: decrease col
        [0, 1]    # RIGHT: increase col
    ])
    
    # Compute next state
    next_y = state_y + action_deltas[action, 0]
    next_x = state_x + action_deltas[action, 1]
    
    # Lookup distance from next state to goal (distances captured from global scope)
    dist = distances[goal_y, goal_x, next_y, next_x]
    
    # Q-value = -(distance + 1) where +1 is action cost
    return -(dist + 1.0)

In [241]:
@memo
def grid_agent[gr: grid, a: Actions](init_loc, beta):
    agent: knows(gr)
    agent: chooses(goal in Goals, wpp=1)
    agent: chooses(
        action in Actions, 
        wpp=exp(beta * q_value(init_loc[0], init_loc[1], goal, action))
    )
    return Pr[agent.action == a]

In [242]:
# Visualize the grid and goals
print("="*50)
print("GRID WORLD (0=free, 1=wall)")
print("="*50)
for i, row in enumerate(grid):
    print(f"Row {i}: {row}")

print(f"\nGoals: {[tuple(g) for g in goal_coords]}")
print(f"Start location: [0, 0]")
print("="*50)

# Get probability for all actions
result = grid_agent(init_loc=np.array([0,0]), beta=1.0)

print("\nACTION PROBABILITIES from [0,0] (beta=1.0)")
print("="*50)
for i, action in enumerate(Actions):
    prob = float(result[0, i])
    bar = "█" * int(prob * 50)  # Visual bar
    print(f"{action.name:5s}: {prob:.4f} {bar}")

print(f"\nSum of probabilities: {float(result[0].sum()):.4f}")

# Try different beta values
print("\n" + "="*50)
print("EFFECT OF BETA (rationality parameter)")
print("="*50)
for beta_val in [0.1, 0.5, 1.0, 2.0, 5.0]:
    result_beta = grid_agent(init_loc=np.array([0,0]), beta=beta_val)
    probs = [float(result_beta[0, i]) for i in range(len(Actions))]
    max_action = Actions(np.argmax(result_beta[0]))
    print(f"β={beta_val:4.1f}: {max_action.name:5s} is preferred with P={max(probs):.3f}")
    print(f"       [STAY:{probs[0]:.3f}, UP:{probs[1]:.3f}, DOWN:{probs[2]:.3f}, LEFT:{probs[3]:.3f}, RIGHT:{probs[4]:.3f}]")

GRID WORLD (0=free, 1=wall)
Row 0: [0 0 0 0 0]
Row 1: [0 1 1 0 1]
Row 2: [0 0 0 0 0]
Row 3: [0 1 0 1 0]
Row 4: [0 1 0 1 0]

Goals: [(Array(2, dtype=int32), Array(2, dtype=int32)), (Array(2, dtype=int32), Array(4, dtype=int32)), (Array(4, dtype=int32), Array(2, dtype=int32))]
Start location: [0, 0]

ACTION PROBABILITIES from [0,0] (beta=1.0)
STAY : 0.1852 █████████
UP   : 0.0000 
DOWN : 0.0000 
LEFT : 0.3116 ███████████████
RIGHT: 0.5033 █████████████████████████

Sum of probabilities: 1.0000

EFFECT OF BETA (rationality parameter)
β= 0.1: RIGHT is preferred with P=0.352
       [STAY:0.318, UP:0.000, DOWN:0.000, LEFT:0.330, RIGHT:0.352]
β= 0.5: RIGHT is preferred with P=0.425
       [STAY:0.258, UP:0.000, DOWN:0.000, LEFT:0.318, RIGHT:0.425]
β= 1.0: RIGHT is preferred with P=0.503
       [STAY:0.185, UP:0.000, DOWN:0.000, LEFT:0.312, RIGHT:0.503]
β= 2.0: RIGHT is preferred with P=0.601
       [STAY:0.081, UP:0.000, DOWN:0.000, LEFT:0.317, RIGHT:0.601]
β= 5.0: RIGHT is preferred with P=0

In [243]:
# Debug: Check Q-values for ALL goals, not just goal 0
print("="*50)
print("DEBUGGING: Q-values for each action toward EACH goal")
print("="*50)

for goal_idx in range(len(goal_coords)):
    goal = goal_coords[goal_idx]
    print(f"\nGoal {goal_idx} at {tuple(goal)}:")
    for action in Actions:
        q = q_value(0, 0, goal_idx, int(action))
        print(f"  {action.name:5s}: Q = {q:.1f}")
    
    # Show softmax for this specific goal
    qs = np.array([q_value(0, 0, goal_idx, int(a)) for a in Actions])
    probs = np.exp(1.0 * qs) / np.sum(np.exp(1.0 * qs))
    print(f"  Softmax (β=1.0): {dict(zip([a.name for a in Actions], [f'{p:.3f}' for p in probs]))}")

print("\n" + "="*50)
print("MARGINALIZED ACTION PROBABILITIES (current model)")
print("="*50)
print("This is P(action) = Σ_goal P(goal) · P(action|goal)")
print("With uniform prior P(goal) = 1/3 for each goal\n")

result = grid_agent(init_loc=np.array([0,0]), beta=1.0)
for i, action in enumerate(Actions):
    prob = float(result[0, i])
    print(f"{action.name:5s}: {prob:.4f}")

print("\n" + "="*50)
print("SOLUTION: Condition on a specific goal instead")
print("="*50)
print("For forward planning, you want P(action|goal), not P(action)")
print("\nExample: If agent is pursuing goal 0 at [2,2]:")
qs_goal0 = np.array([q_value(0, 0, 0, int(a)) for a in Actions])
probs_goal0 = np.exp(1.0 * qs_goal0) / np.sum(np.exp(1.0 * qs_goal0))
for action, prob in zip(Actions, probs_goal0):
    print(f"  {action.name:5s}: {prob:.4f}")
print("\nNow DOWN has 32% probability because we conditioned on goal 0!")

DEBUGGING: Q-values for each action toward EACH goal

Goal 0 at (Array(2, dtype=int32), Array(2, dtype=int32)):
  STAY : Q = -5.0
  UP   : Q = -5.0
  DOWN : Q = -4.0
  LEFT : Q = -5.0
  RIGHT: Q = -6.0
  Softmax (β=1.0): {'STAY': '0.164', 'UP': '0.164', 'DOWN': '0.447', 'LEFT': '0.164', 'RIGHT': '0.060'}

Goal 1 at (Array(2, dtype=int32), Array(4, dtype=int32)):
  STAY : Q = -7.0
  UP   : Q = -7.0
  DOWN : Q = -6.0
  LEFT : Q = -5.0
  RIGHT: Q = -6.0
  Softmax (β=1.0): {'STAY': '0.067', 'UP': '0.067', 'DOWN': '0.183', 'LEFT': '0.498', 'RIGHT': '0.183'}

Goal 2 at (Array(4, dtype=int32), Array(2, dtype=int32)):
  STAY : Q = -7.0
  UP   : Q = -7.0
  DOWN : Q = -6.0
  LEFT : Q = -7.0
  RIGHT: Q = -8.0
  Softmax (β=1.0): {'STAY': '0.164', 'UP': '0.164', 'DOWN': '0.447', 'LEFT': '0.164', 'RIGHT': '0.060'}

MARGINALIZED ACTION PROBABILITIES (current model)
This is P(action) = Σ_goal P(goal) · P(action|goal)
With uniform prior P(goal) = 1/3 for each goal

STAY : 0.1852
UP   : 0.0000
DOWN : 0.